# axon-lang — treinar tudo do zero

Uma passada só: apaga os experts antigos, treina os **18** de novo, valida, mede o
`alpha` e responde algumas perguntas de exemplo.

Feito pra rodar com **Ambiente de execução → Executar tudo** (`Ctrl+F9`).

> **Antes:** `Ambiente de execução → Alterar o tipo de ambiente → T4 GPU`.
>
> A célula 2 vai pedir autorização pra montar o Google Drive e fica esperando aí até
> você autorizar.
>
> **Esta execução apaga `MyDrive/axon_experts/` inteiro.** É o que foi pedido — mas se
> mudou de ideia, ponha `APAGAR_TUDO = False` na célula 2 antes de rodar.

São 18 experts porque `bash` e `shell` viram um só, o `terminal`: separados eles
disputavam o mesmo vocabulário e o `bash` roteava a 62%.

## 1. Ambiente — clonar, compilar com CUDA, importar

In [ ]:
CUDA = True

%cd /content
import os, subprocess, sys

REPO = "https://github.com/geraldogrise/axon-llm.git"
if os.path.isdir("axon-llm"):
    !git -C axon-llm pull --quiet && echo "repo atualizado"
else:
    !git clone --depth 1 $REPO axon-llm

!apt-get -qq install -y ninja-build > /dev/null
!pip -q install pybind11 numpy

import pybind11
cfg = ["cmake", "-S", ".", "-B", "build-colab", "-G", "Ninja",
       "-DCMAKE_BUILD_TYPE=Release",
       "-DAXON_BUILD_PYTHON=ON", "-DAXON_BUILD_TESTS=OFF", "-DAXON_BUILD_EXAMPLES=OFF",
       "-DAXON_ENABLE_NATIVE=OFF",
       f"-DAXON_ENABLE_CUDA={'ON' if CUDA else 'OFF'}",
       f"-DAXON_USE_CUBLAS={'ON' if CUDA else 'OFF'}",
       f"-Dpybind11_DIR={pybind11.get_cmake_dir()}"]
for c in (cfg, ["cmake", "--build", "build-colab", "-j"]):
    p = subprocess.run(c, cwd="axon-llm", capture_output=True, text=True)
    print(p.stdout[-600:] or p.stderr[-600:])
    assert p.returncode == 0, "build falhou -- veja o log acima"

sys.path.insert(0, "/content/axon-llm/notebooks")
sys.path.insert(0, "/content/axon-llm/python")
import pyaxon as ax
import axon_colab as ac

print("\npyaxon ok | cuda:", ax.cuda_available(),
      "|", ax.cuda_device_name() if ax.cuda_available() else "sem GPU")

## 2. Apagar os experts antigos

Monta o Drive (vai pedir autorização) e limpa `MyDrive/axon_experts/`. Só os experts —
não toca em mais nada do seu Drive.

In [ ]:
import shutil

APAGAR_TUDO = True

RAIZ = ac.drive_dir("axon_experts")     # monta o Drive se preciso

antigos = sorted(d for d in os.listdir(RAIZ) if os.path.isdir(os.path.join(RAIZ, d)))
print(f"{len(antigos)} experts hoje: {antigos}")

if APAGAR_TUDO:
    for d in antigos:
        shutil.rmtree(os.path.join(RAIZ, d))
    print(f"\napagados. sobrou: {sorted(os.listdir(RAIZ))}")
else:
    print("\nAPAGAR_TUDO=False -- nada foi removido")

## 3. Treinar os 18

Um expert por vez, salvando cada um no Drive antes de começar o próximo. Se a sessão
cair, rode de novo com `APAGAR_TUDO = False` na célula 2: a fila pula o que já está
salvo e continua de onde parou.

Um expert que falhar não derruba a fila — o erro é registrado e ela segue.

In [ ]:
resumo = ac.treinar_fila(ac.ORDEM_FINAL, repo="axon-llm",
                         extra_env={"AXON_EPOCHS": "300"})

## 4. Validar o gate de domínio

Passa as 266 perguntas de teste pelo sistema completo. Cada pergunta tem que achar o
expert certo entre os 18 — sem saber de qual ela veio.

É a peça que só existe com vários experts: se o gate errar o expert, a resposta certa
fica inalcançável por melhor que aquele expert seja.

In [ ]:
import collections

sistema = ax.system.AxonSystem.load(RAIZ)
carregados = {e.name for e in sistema.experts}
print(f"{len(carregados)} experts carregados\n")
for e in sorted(sistema.experts, key=lambda x: x.name):
    print(f"  {e.name:<22} {len(e.kb.texts):>6} passagens")

acertos, totais = collections.Counter(), collections.Counter()
confusao = collections.Counter()

for nome in sorted(ac.EXPERTS):
    alvo = ac.SAIDA[nome]
    if alvo not in carregados:
        continue
    for _, q in ac.perguntas(nome, repo="axon-llm"):
        escolhido, _ = sistema.route(q)
        totais[alvo] += 1
        if escolhido.name == alvo:
            acertos[alvo] += 1
        else:
            confusao[(alvo, escolhido.name)] += 1

tot, ok = sum(totais.values()), sum(acertos.values())
print(f"\ngate de domínio: {ok}/{tot} = {ok / tot:.1%}\n")

for alvo in sorted(totais, key=lambda a: acertos[a] / totais[a]):
    print(f"  {alvo:<22} {acertos[alvo]:>3}/{totais[alvo]:<3} "
          f"{acertos[alvo] / totais[alvo]:>6.0%}")

print("\nconfusões mais comuns:")
for (a, b), n in confusao.most_common(10):
    print(f"  {a:<22} -> {b:<22} {n}")

## 5. Medir o `alpha`

O `route()` mistura dois sinais: a recuperação e um classificador de palavras. O `alpha`
pesa os dois — `0` é só recuperação, `1` é só o classificador, `0.5` é o padrão.

Não retreina nada, só re-roteia as mesmas perguntas cinco vezes.

In [ ]:
testes = [(ac.SAIDA[n], q) for n in sorted(ac.EXPERTS)
          if ac.SAIDA[n] in carregados
          for _, q in ac.perguntas(n, repo="axon-llm")]

melhor = (0, None)
for alpha in (0.0, 0.25, 0.5, 0.75, 1.0):
    ok = sum(sistema.route(q, alpha=alpha)[0].name == alvo for alvo, q in testes)
    print(f"alpha={alpha:<5} {ok}/{len(testes)} = {ok / len(testes):.1%}")
    melhor = max(melhor, (ok, alpha))

print(f"\nmelhor: alpha={melhor[1]}")

## 6. Perguntar ao sistema

`answer()` faz o caminho completo: escolhe o expert, recupera as passagens e monta o
texto. O `mode` diz como a resposta saiu:

- `extractive` — o trecho da lição, quase cru. É o que sai aqui: não há LLM rodando.
- `generated` — resposta redigida. Exige o modelo do `finetune_expert_colab.ipynb`.
- `abstain` — nada relevante o bastante, e ele diz isso em vez de inventar.

In [ ]:
for pergunta in ["como usar goroutines e canais?",
                 "o que é o determinante de uma matriz?",
                 "como escrever um dockerfile?",
                 "como buscar texto com grep?",
                 "o que é o ownership em Rust?"]:
    r = sistema.answer(pergunta)
    print(f"P: {pergunta}")
    print(f"   [{r['expert']} · {r['mode']}]")
    print(f"R: {r['answer'][:350]}")
    print()

## Pronto

Os 18 experts estão em `MyDrive/axon_experts/` e sobrevivem à sessão cair.

O sistema já **acha** o material certo. O que falta é ele **escrever** a resposta em vez
de devolver o trecho da lição — isso é o `finetune_expert_colab.ipynb`, o QLoRA sobre o
DeepSeek-Coder e o Qwen2.5-Coder.